In [2]:
pip install openeo

Defaulting to user installation because normal site-packages is not writeable
  Using cached openeo-0.45.0-py3-none-any.whl.metadata (8.2 kB)
  Using cached xarray-2025.1.1-py3-none-any.whl.metadata (11 kB)
  Using cached pystac-1.14.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached Deprecated-1.2.18-py2.py3-none-any.whl.metadata (5.7 kB)
  Using cached oschmod-0.3.12-py2.py3-none-any.whl.metadata (10.0 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached openeo-0.45.0-py3-none-any.whl (335 kB)
Using cached Deprecated-1.2.18-py2.py3-none-any.whl (10.0 kB)
Using cached oschmod-0.3.12-py2.py3-none-any.whl (14 kB)
Using cached pystac-1.14.1-py3-none-any.whl (207 kB)
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.7 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.7 MB ? eta -:--:--
   ------------ --

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [1]:
import openeo
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import xarray as xr


In [2]:
connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()
print("Connected to Copernicus Data Space Ecosystem")

Visit https://identity.dataspace.copernicus.eu/auth/realms/CDSE/device?user_code=FELL-BSZR 📋 to authenticate.

❌ Timed out

OidcDeviceCodePollTimeout: Timeout (300.0s) while polling for access token.

In [11]:
aoi = {
  "type": "FeatureCollection",
  "features": [
    {
      "type": "Feature",
      "properties": {},
      "geometry": {
        "coordinates": [
          [
            [
              113.13752171305242,
              -8.043449439076795
            ],
            [
              113.13752171305242,
              -8.231905093756396
            ],
            [
              113.33911043324912,
              -8.231905093756396
            ],
            [
              113.33911043324912,
              -8.043449439076795
            ],
            [
              113.13752171305242,
              -8.043449439076795
            ]
          ]
        ],
        "type": "Polygon"
      }
    }
  ]
}

# Define spatial extent from AOI coordinates
spatial_extent = {
    "west": 113.13752171305242, 
    "south": -8.231905093756396, 
    "east": 113.33911043324912, 
    "north": -8.043449439076795
}

start_date = "2025-09-20"
end_date = "2025-10-21"

print(f"AOI defined for coordinates: {spatial_extent}")
print(f"Time range: {start_date} to {end_date}")
print("Setup completed successfully")

AOI defined for coordinates: {'west': 113.13752171305242, 'south': -8.231905093756396, 'east': 113.33911043324912, 'north': -8.043449439076795}
Time range: 2025-09-20 to 2025-10-21
Setup completed successfully


In [15]:
print("Loading Sentinel-5P NO2 data...")

s5p_no2 = connection.load_collection(
    "SENTINEL_5P_L2",
    temporal_extent=[start_date, end_date],
    spatial_extent=spatial_extent,
    bands=["NO2"],
)

s5p_monthly = s5p_no2.aggregate_temporal_period(
    period="day",
    reducer="mean"
)
# s5p_monthly = s5p_no2.aggregate_spatial(reducer="mean", geometries=aoi)

print("Data collection and aggregation configured successfully")

Loading Sentinel-5P NO2 data...
Data collection and aggregation configured successfully


In [16]:
print("Starting data processing job...")

job = s5p_monthly.execute_batch(
    title="NO2", 
    outputfile="no2_averages_1month.nc"
)


Starting data processing job...
0:00:00 Job 'j-25102305381348f8afd0a94af81cd6f4': send 'start'
0:00:13 Job 'j-25102305381348f8afd0a94af81cd6f4': created (progress 0%)
0:00:18 Job 'j-25102305381348f8afd0a94af81cd6f4': created (progress 0%)
0:00:25 Job 'j-25102305381348f8afd0a94af81cd6f4': created (progress 0%)
0:00:33 Job 'j-25102305381348f8afd0a94af81cd6f4': created (progress 0%)
0:00:43 Job 'j-25102305381348f8afd0a94af81cd6f4': created (progress 0%)
0:00:56 Job 'j-25102305381348f8afd0a94af81cd6f4': running (progress N/A)
0:01:12 Job 'j-25102305381348f8afd0a94af81cd6f4': running (progress N/A)
0:01:31 Job 'j-25102305381348f8afd0a94af81cd6f4': running (progress N/A)
0:01:56 Job 'j-25102305381348f8afd0a94af81cd6f4': finished (progress 100%)


In [8]:
# Buka file NetCDF
ds = xr.open_dataset("data_kedua_NO2.nc")

# Ubah ke DataFrame
df = ds.to_dataframe().reset_index()

# Simpan ke CSV
df.to_csv("data_NO2_kedua.csv", index=False)

print("Konversi selesai! File tersimpan sebagai data.csv")


Konversi selesai! File tersimpan sebagai data.csv
